# 第9回: Human-in-the-Loop, Sandbox

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session09/session09_hitl_sbx.ipynb)

AI は大量の情報処理を高速にこなせる一方、曖昧な状況の理解や価値判断、高い影響を伴う最終判断には限界がある。自律性を活かしながらこの限界を補うには、人間の判断を処理フローに組み込む仕組みと、実行環境そのものを隔離する仕組みが必要になる。

今回は、次の2つを学ぶ。

- **Human-in-the-Loop（HITL）**: 人間の判断を AI の処理フローへ意図的に組み込む設計パターン
- **サンドボックス**: コマンドに許す権限と資源を制限し、問題が起きても影響範囲を閉じ込める

---

# 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session09

%pip install -q -e ".[server]"
!curl -fsSL https://chatgpt.com/codex/install.sh | CODEX_NON_INTERACTIVE=1 sh

---

## 1. Human-in-the-Loopの概要

Human-in-the-Loop（HITL）は、AI の処理フローへ人間の判断を意図的に組み込む設計パターン。AI は大量の情報処理や案の生成を高速に行える一方、曖昧な状況の理解、価値判断、説明責任、高い影響を伴う最終判断には限界がある。


人間を介在させる条件は、操作の影響度、可逆性、モデルの確信度、組織のポリシーなどから決める。低リスクな読み取り操作まで毎回承認させると、待ち時間とレビュー負荷が増え、承認が形骸化しやすい。

### HITLのユースケース

HITL は、AI の判断だけで処理を完結させることが適切でない場面で利用する。コンテンツモデレーション、カスタマーサポート、金融、法務、生成物の品質管理など、さまざまな領域で「どの判断を自動化し、どの条件で人間へ戻すか」を設計する。

| 活用パターン | 人間の役割 | ユースケース例 |
|---|---|---|
| 承認・拒否 | 高リスクな操作の実行可否を決める | ファイル削除、送信、決済 |
| 修正 | AI の案や操作引数を書き換える | 回答案の編集、保存先の変更 |
| エスカレーション | AI の能力や権限を超える案件を引き取る | 複雑な問い合わせ、例外処理 |
| 意思決定支援 | AI の分析を材料に人間が最終判断する | 審査、診断支援、法務レビュー |
| フィードバック | 評価や訂正を将来の改善に利用する | アノテーション、選好データ収集 |

> **Human-on-the-Loop（HOTL）との違い**: HITL では、特定の判断や操作の前に処理を止め、人間の判断を実行フローへ組み込む。HOTL では、AI が通常は個々の判断で処理を止めずに自律動作し、人間はその動作や結果を監督する（例: ログ監視、品質レビュー）。必要に応じて、実行中の停止や是正、成果物の事後修正、ルールや設定の変更などで介入する。

---

## 2. 承認の実装

### 承認が有効な操作

HITL の中でも、承認が特に有効なのは、エージェントが外部世界へ変更を加える直前。

- ファイルの作成・上書き・削除
- メールやメッセージの送信
- データベースの更新
- 決済、発注、公開、デプロイ
- コマンドや外部ツールの実行

承認時には、操作の内容やリスクに応じて、対象、変更内容、実行理由、想定される影響などを提示し、人間が実行可否を判断できるようにする。

### 承認の設計ポイントと限界

- 承認点はリスクに応じて絞り込む

    人間のレビューは遅く、スケールしにくい。大量の承認要求は「内容を読まずに承認する」状態を招き、人間が承認しても判断そのものを誤ることもある。
- レビューした内容と実行する内容を一致させる

    承認後に引数やプロンプトを組み立て直すと、レビュー対象と実行対象がずれる。承認された値をそのまま後続処理へ渡し、承認後に変更が入らないようにする。
- 承認要求そのものが攻撃面になる

    プロンプトインジェクションで承認を促す文面を作る、無害な要求を大量に混ぜて危険な1件を埋没させる、見た目が似た文字で宛先やパスを偽装するといった手口がある。承認だけに頼らず、許可リストや上限値などの機械的なポリシー検査と併用する。
- 承認が得られない場合の既定動作を決める

    タイムアウトや担当者不在で「既定で許可」に倒すと、承認は形だけになる。既定は保留または拒否とする。また、中断中も外部の状態は変化するため、待ち時間が長い承認には有効期限を設け、再開時に前提を検証する。

### LangGraphによる承認フロー

LangGraph の `interrupt()` はノードの実行を一時停止し、レビューに必要な情報を呼び出し側へ返す。人間の判断を `Command(resume=...)` で渡すと、同じ `thread_id` のチェックポイントから処理を再開できる。

ここでは、ファイルへの書き込み案を作り、承認された場合だけ `apply_write` ノードへ進む。ノードへ到達したことは完了メッセージで確認し、実際のファイル書き込みは行わない。


LLM は使わず、承認制御だけに注目する。実際のエージェントでは、最初の「書き込み案」を LLM やツール呼び出しが生成する。また、`apply_write`では実際に書き込まずに完了メッセージを表示して処理を終了する。

[![](https://mermaid.ink/img/pako:eNp1Uk1PwkAQ_SvNnGpSCGD6tQcTIx71oJxkCdm0A622u82wBZHy390Wqa3CnObtvjf73mQPEKkYgcEqU7soEaSt2ZQTl5apjTbYnr_O7l9mi5vzaUFYCMLljlKNdg-1nKTMhVwSblPc2V3QMkRRZPufGZ2-vSeMFMVG9Y6RTpW0_x60zCXK2J4_Pk9PFnvmrcHgrm_4YoqG1rV5KUdDqjjMa--ktmhVFsapthYcqm6e6-KT-epfuAtLaQR1tGsL6RF-Q-8z7D-8SrOMEbY0cGBNaQxMU4kO5Ei5qCEc6hEcdII5cmCmjQV9cODyaDSFkG9K5WcZqXKdnEFZxELjNBVrEoaxEtmmphhvSA-qlBpYeOs3M4Ad4NPAcDhyA3ccemM_cEe-58AemDsZhqb8iRd6wdgNJ0cHvppHR8PAdx2o163o6fRjm497_AZhue6u?type=png)](https://mermaid.live/edit#pako:eNp1UstOwzAQ_JVoT0FKqz6U5wEJUY5wgJ6oq8qKt40hiaPFaSlN_x0npWkCZU879sx6ZuUDxEogRLBO1S5OOGlrPmPEcsvUhzbYXrzM757ny5vzaUFYcMLVjqRGu4daTlJmPF8RbiXu7C5oGbwo0v3PjE7f3hPGioRRvWGspcrt3wctc4W5sBcPT7OTxZ55azC47Ru-mqKhdW1ey9GQKgaL2jupLVqVhULqJYOqG-d_7cl79SfblZ00gjrZf_voES6Z9yn2H17LNI0IWxo4sCEpINJUogMZUsZrCId6BAOdYIYMItMKTu8MWH40moLnr0plZxmpcpOcQVkIrnEm-Yb4hWGsId2rMtcQhdNpMwKiA3waGA5HbuCOQ2_sB-7I9xzYQ-ROhqEpf-KFXjB2w8nRga_mzdEw8F0H6mUrejz91-bbHr8Bzu_uPw)

In [ ]:
import json
from typing import Literal, TypedDict
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

# State
class FileApprovalState(TypedDict, total=False):
    path: str
    content: str
    approved_content: str
    decision: Literal['approve', 'reject']
    review_note: str
    result: str

# Nodes
def prepare_write(state: FileApprovalState) -> dict:
    if not state['content']:
        raise ValueError('空の内容は書き込めません')
    return {'approved_content': state['content']}

def human_review(state: FileApprovalState) -> dict:
    # interrupt より前では副作用を起こさない
    response = interrupt(
        {
            'action': 'write_file',
            'path': state['path'],
            'preview': state['content'],
            'allowed_decisions': ['approve', 'edit', 'reject'],
        }
    )

    decision_type = response.get('type')
    if decision_type == 'approve':
        return {
            'decision': 'approve',
            'approved_content': state['content'],
            'review_note': '提案どおり承認',
        }
    if decision_type == 'edit':
        edited_content = response.get('content')
        if not isinstance(edited_content, str) or not edited_content:
            raise ValueError('edit では空でない content が必要です')
        return {
            'decision': 'approve',
            'approved_content': edited_content,
            'review_note': '人間が内容を修正して承認',
        }
    if decision_type == 'reject':
        return {
            'decision': 'reject',
            'review_note': str(response.get('message', '理由なし')),
        }
    raise ValueError(f'未対応の判断です: {decision_type}')


def route_after_review(state: FileApprovalState) -> str:
    return 'record_rejection' if state['decision'] == 'reject' else 'apply_write'


def apply_write(state: FileApprovalState) -> dict:
    return {'result': f'書き込み完了: {state['path']}'}


def record_rejection(state: FileApprovalState) -> dict:
    return {'result': f"書き込み中止: {state['review_note']}"}

In [ ]:
approval_builder = StateGraph(FileApprovalState)
approval_builder.add_node('prepare_write', prepare_write)
approval_builder.add_node('human_review', human_review)
approval_builder.add_node('apply_write', apply_write)
approval_builder.add_node('record_rejection', record_rejection)

approval_builder.add_edge(START, 'prepare_write')
approval_builder.add_edge('prepare_write', 'human_review')
approval_builder.add_conditional_edges(
    'human_review',
    route_after_review,
    {
        'apply_write': 'apply_write',
        'record_rejection': 'record_rejection',
    },
)
approval_builder.add_edge('apply_write', END)
approval_builder.add_edge('record_rejection', END)

# 中断・再開にはチェックポインタが必要
approval_graph = approval_builder.compile(checkpointer=InMemorySaver())

In [ ]:
approval_config = {
    'configurable': {'thread_id': f'file-write-{uuid4()}'}
}
proposal = {
    'path': 'reports/summary.md',
    'content': '# 実行結果\n\n承認後に書き込む想定の内容です。\n',
}

# human_review で停止する
pending = approval_graph.invoke(proposal, config=approval_config)
review_request = pending['__interrupt__'][0].value
print(json.dumps(review_request, ensure_ascii=False, indent=2))

In [ ]:
# 同じ thread_id に人間の判断を渡して再開する
approved = approval_graph.invoke(
    Command(resume={'type': 'approve'}),
    config=approval_config,
)

print(approved['result'])

In [ ]:
# 別スレッドで拒否の経路も確認する
reject_config = {
    'configurable': {'thread_id': f'file-write-{uuid4()}'}
}
rejected_proposal = {
    'path': 'reports/delete-me.md',
    'content': 'この内容は保存されません。\n',
}

# 実行
approval_graph.invoke(rejected_proposal, config=reject_config)

# 再開
rejected = approval_graph.invoke(
    Command(
        resume={
            'type': 'reject',
            'message': '保存先と内容を再確認するため',
        }
    ),
    config=reject_config,
)

print(rejected['result'])

### 実装の要点

- **`interrupt()`**: JSON シリアライズ可能なレビュー情報を返し、グラフを停止する
- **`InMemorySaver`**: `thread_id` ごとの状態を保持する。教材には便利だが、プロセス終了で消えるため、本番ではデータベース等の永続チェックポインタへ置き換える
- **`Command(resume=...)`**: 人間の判断を `interrupt()` の戻り値として渡し、同じスレッドを再開する
- **副作用の分離**: `human_review` ノードは停止・再開時に先頭から再実行されるため、副作用を伴う実処理は後続ノードに置く。このデモの `apply_write` は完了メッセージだけを返す

`edit` を試す場合は、再開時に `{'type': 'edit', 'content': '修正後の内容'}` を渡す。人間が編集した内容が `approved_content` として固定され、後続ノードへ渡される。

---

### ReActグラフのリファクタリング

これまでは、モデルノード、ツールノード、条件分岐を接続し、ReAct ループを自前で組み立てた。ノードと辺を直接書く方法は任意のワークフローを表現できるが、その柔軟性が生きるのは構造そのものを設計する場合。ReAct のように構造が標準化されたループでは、ツール呼び出しの検出、実行結果のメッセージ履歴への追加、ループ終了の判定がどれも定型処理になり、自前で書いても得られる利点は小さい。記述量が増え、実装ごとの差異や抜け漏れが入り込む余地だけが残る。

ここからはコードの見通しをよくするため、ReAct ループの構築を `create_agent()` に任せる。`create_agent()` は、コンパイル済みのReActループグラフを生成する。

In [ ]:
# @title create_agentによるReActグラフの生成
from IPython.display import Image, display
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.runnables.graph_mermaid import CurveStyle
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel

@tool
def fake_tool(input: str) -> str:
    """ダミーのツール"""
    return f'fake_tool: {input}'

fake_model = FakeMessagesListChatModel(responses=[])

react_agent = create_agent(
    model=fake_model,
    tools=[fake_tool],
)

display(
    Image(
        react_agent.get_graph().draw_mermaid_png(curve_style=CurveStyle.NATURAL)
    )
)

#### Middleware（フック）の仕組み

`create_agent()` は、モデルやツールの前後へ処理を差し込む Middleware を受け取る。Middleware は `before_agent`、`after_agent`、`before_model`、`after_model`、`wrap_model_call`、`wrap_tool_call` などのフックを通じて、プロンプトの変更、ログ記録、再試行、ツール実行の制御といった横断的な処理を ReAct ループへ追加する。

[![](https://mermaid.ink/img/pako:eNqdVFFvmzAQ_ivW7aWVSBTSkARrmrQ2fdwe1jxtTJEbTIJicGSM2izKf98BgRyMVO14AO78fd_5u5N9hLUOJXCIlH5Zb4WxbLkITJAyfJ5lpI1cJYhQNwHQMIDbGpVZZN38CuBp-fXHMoDfzUpN7DKs1irDfPkl-ZVMw0Lo8fuCypzrio1M7WUbZUjIIrLSNCASEUwbW2-PRGdsYy1_3hix37IXfFeI1VooVa83JusEGrjOL-w2dCoRxkaubaxTdr-k-ao_benaRqtMMQE2GHxptaprmq5RcMtA3-gRjOgrLeglfL4wWqYpoaNXFaGj6MM3aucSb-DJErtAeztz7UsopcSqMwM6C5rL7EHJfwyaV-6OHXMo3ggxeicHocjwzBlx4MxjXg__YpjS38FuzaMX77AoVoqzT5PJpJdbGf8gl7b9v6gfrKridPdU0u8cjyXC7KQZ4Jx4qlPZHRI4sDFxCNyaXDqQSJOIIoRjhcQraSsTGQDH3xC1AigWTsjbi_Sn1klNNTrfbOsg34fCykUs8LAjIhIqKyC4C2kedJ5a4BP3zitFgB_hFbjvD0fe3HP9qTube6PZ1IEDcG889PGZjaf-dO56_vjkwJ-y6mg4n6GADGOrzbfqyi5v7tNf3f3eqw?type=png)](https://mermaid.live/edit#pako:eNqdVFFvmzAQ_ivW7aWVSBTSkARrmrQ2fdwe1jxtTJEbTIJicHQYtVmU_74DAjGMVO38YLi777vzdyf7CGsdSuAQKf2y3go0bLkIMEgZrWcZaZSrhBDqJgDbDOC2RmWGWDe_Anhafv2xDOB3E6mJXYbRWmXkL7-WfyXTsEj0-H1hpznXFRuZmssxStMii8hIbECWZWHa2Pp4lnXGNtLy5w2K_Za90F4hVmuhVB1vRNYOEnCdX8ht6HaKMEa5NrFO2f3S9lf9aaeuZbTKFBNgg8GXVqu6ou2YDW4J6Bs9gQl9pQW9hM8XRku0Tejkq4rYo-jDN9nOJd7AWyF2gfZ25trXopQpVp0Z2LOwfZk5KPmPQHzl7tjBQ7ETBPVODkKR0Z1DceDMY14P_yLYpr-D3ZpHL95hUawUZ58mk0kvtxL-Qa7d9v-ifrCqitPdU0m_czyWCNxJHNCceKpT2R0SOLDBOARuMJcOJBITUZhwrJD0JG1lIgPg9BtSrgCKwIl4e5H-1DqpqajzzRZ4JFRGVr4PhZGLWNBtTxov0jEkPug8NcAn7t2ozAL8CK_AfX848uae60_d2dwbzaYOHIB746FPazae-tO56_njkwN_yrKj4XzmOSDD2Gj8Vr3Z5dN9-gu7l97x)

`HumanInTheLoopMiddleware` は `after_model` フックを利用する。モデルが返した `AIMessage` のツール呼び出しを調べ、`interrupt_on` の対象であれば、`tools` ノードへ進む前に `interrupt()` でグラフを中断する。再開時に渡された `approve`、`edit`、`reject` の判断をツール呼び出しへ反映してから処理を続ける。

今回は、外部へ変更を加える `write_file` を承認対象にし、読み取り専用の `read_file` はそのまま実行する。

In [ ]:
# @title HumanInTheLoopMiddlewareの設定
from langchain.agents.middleware import HumanInTheLoopMiddleware

hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        'write_file': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'file_delete': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'read_file': False,
    },
    description_prefix='Toolの実行には承認が必要',
)

In [ ]:
# @title APIキーの設定
import getpass
import os

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY を入力する: ')

# guarded_agent の web_search / fetch_url は import 時に Tavily を初期化する
if not os.environ.get('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass.getpass('TAVILY_API_KEY を入力する: ')

print('OpenAI APIキー設定完了' if os.environ.get('OPENAI_API_KEY') else 'OpenAI APIキー未設定')
print('Tavily APIキー設定完了' if os.environ.get('TAVILY_API_KEY') else 'Tavily APIキー未設定')

#### 長期記憶グラフへ組み込む

- 長期記憶グラフ(`load_memory`, `extract_memory`, `validate_memory`, `write_memory`)に `create_agent()`が返したグラフをノード（サブグラフ）として追加する。
- 外側の長期記憶グラフと内側のReActグラフは状態を共有する必要がある
- `load_memory`で取得した情報をReActの`model`のプロンプトに差し込む(`dynamic_prompt`)


[![](https://mermaid.ink/img/pako:eNp9lFtvmzAUx78Kcl86iURATEPQNKlaJm1Su4c0TysV8rCTWPEFOWYJi_Lda5xCDMlmicuxfz6XPz4cQSExASlYMbkvNkhp72mRqUx4Zuy0se9fM_CyfFwsM_D2qV1hEuGcEy5VfZ8Bx8pAx5CDVqjQF6w_4ZB_EKMYaXJBBzMOu1fUBV3ToXIicJP4t5_zNu2uqOr3WqFy4y3IY6Hb2WZgqkihqRTecu7OWxlyRUzuN8VoBjcqMpOPfTqJNENLyXZmzT4HaybPi2cn2y52m-3e3HPrPS8QY94rrgXitMhLJTkv9XgAvLleugwHoV3TfUcrTVTe1vS94kj8EMsNeZKyfKYYM7JHiowdbFDWP-TzRqMvw0pcdFhkg7tBhqJ6n_9DDJ1Z9vwNbpdqw3UfpDvGF5l6fWFp9-Rft4ZFesfMGuc4_Wa42TMWHLbC7Z75UJZece6chfKPgq6Lqhm5Ek0d0jDyVd3cDaLkloww2pkfhUJ16sVe3N_vynkT970VZSz17iCE_a1nbezqXRSZcE7sTlwqti-WjvyJx5HaEjUyBaVCCuJW1FzAB2tFMUi1qogPOFEcNSY4Npzpxw3hJAOpecXGUwYycTJ7SiR-ScnbbUpW601rVGWj-Zwi05SGWCG2axCTAVFfZSU0SMN4Zn2A9AgOxgzjMUzgNJ4mwRRO4MSs1mY6gONZHAUJTKIkCOPw5IO_NmowTh4iOEsCCGcP04c4hD4gmGqpns9_avvDPr0DX3XW_A?type=png)](https://mermaid.live/edit#pako:eNp9lF1vmzAUhv8Kcm86iURATEPQNKlaJm1Su4s0VysV8rCTWPEHcswSFuW_1ziFGJLNUgLn-PH5eOFwBIXEBKRgxeS-2CClvadFpjLhmbXTxr5_zcDL8nGxzMDbp3aHSYRzTrhU9X0GHCsDHUMOWqFCX7C-wyH_IEYx0uSCDjwOu1fUBV3ToXIicFP4t5_ztuyuqer3WqFy4y3IY6Fbb7MwVaTQVApvOXf9VoZcEVP7TTGaxY2KzNRjr04hzdJSsp3Zs9fBnqnzEtmptsvdVrs3_7mNnheIMe8V1wJxWuSlkpyXejwA3twoXYWD1K7p3qOVJipve_pecSR-iOWGPElZPlOMGdkjRcYONmjrH_J5o9GXYScuOmyywd0kQ1G9z_8hhsEse34Gt1u16boH0r3GF5l6c2Fp982_Hg2L9F4za5zz9Ifh5sxYcDgKt2fmQ1l6xbk-C-UfDV03VTNyJZo6pGHkq7r5N4iSWzLCaGc-FArVqRd7cf-8K-dN3PdWlLHUu4MQ9o-etbG7d1Fk0jm5O3Gp2L5YOvInHkdqS9TINJQKKYjbUfMDPlgrikGqVUV8wIniqDHBseHMPG4IJxlIzS02kTKQiZM5UyLxS0reHlOyWm9AukJsZ6yqbESfU2SmkndeZUog6qushAZpGEMbBKRHcDBmGI9hAqfxNAmmcAInMx_Uxh3A8SyOggQmURKEcXjywV-bNhgnDxGcJQGEs4fpQxyacARTLdXz-VNtv9indzZm10I)

In [ ]:
# @title 長期記憶グラフの状態と読み込みノード
from dataclasses import dataclass
from typing_extensions import NotRequired

from langchain.agents.middleware import AgentState
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime


class LongTermMemoryState(AgentState):
    memories: NotRequired[list[str]]
    memory_candidates: NotRequired[list[str]]
    approved_memories: NotRequired[list[str]]

In [ ]:
# @title create_agentのグラフを長期記憶グラフへ追加
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain_openai import OpenAIEmbeddings
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.store.memory import InMemoryStore
from langchain_openai import ChatOpenAI

from guarded_agent.state import Context
from guarded_agent.memory import load_memory, make_extract_memory, validate_memory, write_memory
from guarded_agent.tools import DEFAULT_TOOLS


@dynamic_prompt
def memory_system_prompt(request: ModelRequest) -> str:
    memories = request.state.get('memories', [])
    prompt = """あなたはAIエージェントです。
必要な場合はToolを利用してください。
"""
    if memories:
        memory_text = '\n'.join(f'- {memory}' for memory in memories)
        prompt += f"""
参考になる長期記憶:
{memory_text}
"""
    return prompt

model_id = 'gpt-5.4-nano' #@param ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna', 'gpt-5.5', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']
model = ChatOpenAI(model=model_id)
memory_hitl_agent = create_agent(
    model=model,
    tools=DEFAULT_TOOLS,
    state_schema=LongTermMemoryState,
    middleware=[memory_system_prompt, hitl_middleware],
)

long_term_store = InMemoryStore(
    index={
        'embed': OpenAIEmbeddings(model='text-embedding-3-small'),
        'dims': 1536,
        'fields': ['text'],
    }
)

builder = StateGraph(
    LongTermMemoryState,
    context_schema=Context,
)
builder.add_node('load_memory', load_memory)
builder.add_node('agent', memory_hitl_agent)
builder.add_node('extract_memory', make_extract_memory(model))
builder.add_node('validate_memory', validate_memory)
builder.add_node('write_memory', write_memory)

builder.add_edge(START, 'load_memory')
builder.add_edge('load_memory', 'agent')
builder.add_edge('agent', 'extract_memory')
builder.add_edge('extract_memory', 'validate_memory')
builder.add_edge('validate_memory', 'write_memory')
builder.add_edge('write_memory', END)

graph_with_hitl_tools = builder.compile(
    store=long_term_store,
    checkpointer=InMemorySaver(),
)

display(
    Image(
        graph_with_hitl_tools.get_graph(xray=2).draw_mermaid_png(curve_style=CurveStyle.NATURAL)
    )
)

#### ツール呼び出しを承認する

書き込みを依頼すると、モデルが生成した `write_file` のツール呼び出しを `HumanInTheLoopMiddleware` の `after_model` フックが検出し、ツールの実行前にグラフを中断する。中断情報にはツール名と引数が含まれるため、実行内容を確認してから判断を返す。

ここでは `approve` を返して提案どおり実行する。引数を変更する場合は `edit`、実行しない場合は `reject` を選ぶ。いずれも中断時と同じ `thread_id` で再開する。

In [ ]:
# @title 共通設定
import json
from langgraph.types import Command

hitl_context = Context(user_id='session09-user')


In [ ]:
# @title write_fileの実行
hitl_config = {
    'configurable': {'thread_id': f'hitl-agent-{uuid4()}'}
}

pending_tool_call = graph_with_hitl_tools.invoke(
    {
        'messages': [
            HumanMessage(
                content=(
                    '私の好みは簡潔な回答です。'
                    'この内容をprofile.txtへ保存してください。'
                )
            )
        ]
    },
    config=hitl_config,
    context=hitl_context,
)

review = pending_tool_call['__interrupt__'][0].value
print(json.dumps(review, ensure_ascii=False, indent=2))


In [ ]:
# @title 承認してグラフを再開する
completed_tool_call = graph_with_hitl_tools.invoke(
    Command(
        resume={
            'decisions': [{'type': 'approve'}],
        }
    ),
    config=hitl_config,
    context=hitl_context,
)

completed_tool_call['messages'][-1].pretty_print()

saved_memories = long_term_store.search(
    ('memories', hitl_context.user_id)
)
print('保存された長期記憶:', [item.value['text'] for item in saved_memories])


In [ ]:
# @title file_deleteの実行
from guarded_agent.tools import WORKSPACE_DIR

delete_hitl_config = {
    'configurable': {'thread_id': f'hitl-agent-{uuid4()}'}
}

pending_delete = graph_with_hitl_tools.invoke(
    {'messages': [HumanMessage(content='profile.txtを削除してください。')]},
    config=delete_hitl_config,
    context=hitl_context,
)

review = pending_delete['__interrupt__'][0].value
print(json.dumps(review, ensure_ascii=False, indent=2))

rejected_delete = graph_with_hitl_tools.invoke(
    Command(
        resume={
            'decisions': [{
                'type': 'reject',
                'message': '保存済みのプロファイルは削除しない',
            }],
        }
    ),
    config=delete_hitl_config,
    context=hitl_context,
)

rejected_delete['messages'][-1].pretty_print()

profile_path = WORKSPACE_DIR / 'profile.txt'
print('profile.txt は残っている:', profile_path.exists())


---

## 3. サンドボックス実行

### サンドボックスとは

サンドボックスは、実行するコードやコマンドに与える権限・資源・到達範囲をあらかじめ制限し、問題が起きても影響を内側へ閉じ込める仕組み。

エージェントにコマンド実行やコード実行の Tool を渡すと、実行される内容を事前に確定できなくなる。何が動くかはモデルの出力次第で、そこには次のような経路で危険な操作が混ざる。

- モデルが誤ったコマンドを生成する（対象の取り違え、範囲の広すぎるパス指定）
- 読み込んだ文書や Web ページに含まれるプロンプトインジェクションに従う
- 取得したコードや依存パッケージが、意図しない動作を持ち込む

いずれも悪意ある利用者がいなくても起こる。エージェントが扱う入力の一部は外部から来るため、コマンドが生成されるまでの経路全体を信頼できる前提には置けない。


### 隔離の段階

隔離は 0 か 1 ではなく、強さに段階がある。上ほど軽く、下ほど強い。

| 隔離の方法 | 境界になるもの | 防げないこと |
|---|---|---|
| 作業ディレクトリを変える（`subprocess.run(..., cwd=...)`） | なし（ホストと同じ権限のまま） | 絶対パスでの読み書き、ネットワーク、資源の使い切り |
| 別プロセス＋資源上限（`rlimit`） | プロセスの CPU とメモリ | ファイルアクセス、ネットワーク |
| OS のサンドボックス機構（seccomp、Landlock、Seatbelt） | syscall とファイルパス | カーネルの脆弱性、プロファイルの設定漏れ |
| コンテナ | 名前空間、cgroup、capability | ホストと共有するカーネルの脆弱性 |
| microVM、専用 VM | 仮想化されたカーネル | ハイパーバイザの脆弱性 |
| 別アカウント・別ネットワークの実行環境 | クラウドの信頼境界 | 運用や設定の誤り |

強い隔離ほど起動が重く、内側で使えるツールも減る。第4節で使う `ExecutionPolicy` は、このうち「別プロセス＋資源上限」「OS のサンドボックス機構」「コンテナ」の3段に対応する。

どの段を選んでも、内側で許した操作は起こりうる。そのため、まず何を防ぎたいのかを決めてから制約を選ぶ。



### 脅威モデルから制約を決める

コマンド実行では、少なくとも次の失敗を想定する。

- ホストや他ユーザーのファイルを読み書きする
- 環境変数や認証情報を盗む
- 外部ネットワークへデータを送る、追加のコードを取得する
- CPU、メモリ、プロセス数、ディスクを使い切る
- 長時間終了せず、ワーカーを占有する
- 実行結果を偽装したり、巨大な出力でログ基盤を圧迫したりする

この一覧に対応させて、今回の Docker サンドボックスでは次の制約を固定する。設定はすべて後述の `DockerExecutionPolicy` で指定する。3.1 の段階でいう「コンテナ」に相当するため、他の実行方法を選べば守られる範囲は変わる。違いは「4.1 実装の要点」で比較する。

| 制約 | 設定 | `DockerExecutionPolicy` の指定 |
|---|---|---|
| ホストのファイル | bind mount しない | 作業ディレクトリを渡さない（一時ディレクトリはマウントされない） |
| ルートファイルシステム | 読み取り専用 | `read_only_rootfs=True` |
| 一時書き込み | サイズ制限付き `/tmp` のみ | `extra_run_args` の `--tmpfs` |
| ネットワーク | 無効 | `network_enabled=False` |
| 実行ユーザー | 非 root | `user='65534:65534'` |
| 権限 | capability 全削除、権限昇格禁止 | `extra_run_args` の `--cap-drop=ALL`、`--security-opt no-new-privileges=true` |
| 資源 | メモリ、CPU、プロセス数を制限 | `memory_bytes`、`cpus`、`extra_run_args` の `--pids-limit` |
| 実行時間 | コマンドごとに上限を設ける | `command_timeout` |
| 結果 | 行数とバイト数で切り詰める | `max_output_lines`、`max_output_bytes` |
| ライフサイクル | 実行終了時に破棄 | `remove_container_on_exit=True`（既定） |

裏を返せば、`/tmp` への書き込みとコンテナ内でのプロセス起動は許したままで、その範囲の操作は起こりうる前提で扱う。実行が終わるとコンテナごと破棄されるため、残した書き込みは次の実行へ持ち越されない。この線引きで足りない場合は、3.1 の表でより下の段へ移る。

[![](https://mermaid.ink/img/pako:eNqVlW1vmzAQx78Kct90EokCARLQNKlaJm1SuxdJXq1UyIVLQDE2cpwlLMp3n3EKMQ-tVEuBnP07--5_HJxRzBJAAdoQdoxTzIXxuAx5SA059kLa988hWq0flusQvXypVwjDSZRDznh5HyLNClHDwElwHIsb1p7QyL-YZAkWcEM7Mxp75JkO6qZGRUCTKvAfvxd12E1Sh9ctx0VqLOEhFvVsNZKMQywyRo31Qp9XMkQcZOyDYlTjFTaMQ4S3QIUMa5UCIWvGyFOWJASOmMNYR7RQq5HLGhDppu6dNSF32cs1de-s4Y0A_uGhGtHxlQrdctJ0arKudTrKa6Qii2JMiPGclBTnWRwVnOV5IcYd4EXfpcmuc7Ru9jOq9fh5yDH9RdcpPDJW9NIakuudwhmj0bdWkd4rngI7GelsV40K16PpVs74-gHR3Uyx10IPa6Id18tCm1ZYU-CmIW-ytzpc0XoP95tcIa2GUcb1nHZbD3a_ArtNPdz9bwXIepw-p6DoLaF-UiWBnrb8FFi2ycvqKhHOdjBK8F6-8jguA8M13La_rvogbhqbjJDAuHMcZ8j1WolPurYexU_6XkuiVu9sW2appdzUNKO7laJtc2rkmO-Aj6SOAWUUdCGrHzLRlmcJCgQ_gIly4DmuTHSuOPlKSiGHEAXybyJ3ClFIL9KnwPQPY3ntxtlhm9bGoahKvciwfLfcCBkA8O_sQAUKbN9TW6DgjE4oGE3d8dR3HXc-93zL8ya2iUo5bbnjiTefzHzfnnuOa1nuxUT_1LHW2PZcf-ZLj4nrSGBmIkgywfjT9VunPnmX_1K2Qko?type=png)](https://mermaid.live/edit#pako:eNqVlW1vmzAQx78Kct90EokCAQpomlQtkzap3Yskr1Yq5MIloBgbOWZJVuW7zziFmIdWqqVAzv6dffc_Dl5RwlJAIdoQdkgyzIXxsIx4RA059kLat08RWq3vl-sIPX9pVgjDaVxAwfjpNkKaFaGWgaPgOBFXrDuhkX8xyVMs4Ir2ZjT2wHMd1E2NioGmdeA_fi-asNukqpctx2VmLOE-Ec1sPdKcQyJyRo31Qp9XMsQcZOyjYtTjBTaMQ4y3QIUMa5UBIWvGyGOepgQOmMNUR7RQ61HIGhDppu69NSF32cs1de-t4Y0A_uGhGtHzlQpdc9J0arNudDrIa6wiixNMiPGUnigu8iQuOSuKUkx7wLO-S5td72jdHGbU6PGzKjD9RdcZPDBWDtIak-udwhmTybdOkd4rngJ7GelsX40a16PpV874-gHR30yxl0KPa6IdN8hCm1ZYW-C2Ia-ydzpc0XoPD5tcIZ2GUcblnG5bj3a_AvtNPd79bwXIB5w-p6D4LaFhUicCA235MbRsk5_qq0Q428EkxXv5yuP4FBqu4Xb9ddVHcdPY5ISExo3jOGOul0p80rXzKH7S91IStXpj2zJLLeW2pjndrRRtm3OjwHwHfCJ1DCmjoAtZ_5CJtjxPUSh4BSYqgBe4NtFrzclXUgYFRCiUf1O5U4QiepY-JaZ_GCsaN86qbYbCDSZ7aVVlXetFjuXL5YrICIB_ZxUVKLQDS-2Bwld0ROFk7k7ngeu4vu8FlufNbBOd5LTlTmeeP7sLAtv3HNey3LOJ_qlzrantucFd4AdzZ-57rmsiSHPB-OPlW6c-eef_VPdCWQ)

---

## 4. サンドボックス化したシェルの実装

エージェントのコマンド実行を隔離コンテナへ閉じ込める。

`ShellToolMiddleware` は、永続シェルセッションを `shell` Tool として ReAct ループへ登録する Middleware。`before_agent` でセッションを起動し、`after_agent` で片付ける。セッションが永続するため、`cd` や環境変数の設定が後続のコマンドへ引き継がれる。

コマンドをどこで動かすかは **ExecutionPolicy** で差し替える。Tool の名前と引数（`shell`／コマンド文字列）は変わらないので、エージェントの実装もプロンプトの組み立ても共通のまま、隔離の強さだけを入れ替えられる。

| ExecutionPolicy | 実行場所 | 隔離されるもの | 想定する用途 |
|---|---|---|---|
| `HostExecutionPolicy`（既定） | ホストのプロセス | CPU 時間とメモリのみ（`rlimit`） | すでにコンテナや VM で隔離済みの信頼環境 |
| `CodexSandboxExecutionPolicy` | Codex CLI のサンドボックス | syscall とファイルアクセス | Codex CLI があり、ホスト実行のまま制限を強めたい場合 |
| `DockerExecutionPolicy` | 専用の Docker コンテナ | ファイル、ネットワーク、権限、資源 | 未信頼の入力を扱う場合 |

以下の変更を行う。

1. ExecutionPolicy を設定した `ShellToolMiddleware` を追加する
2. `run_command` と `python_repl` を 除外する
3. `HumanInTheLoopMiddleware` の承認対象に `shell` を加える

In [ ]:
# @title 利用できるExecutionPolicyの確認
import shutil
import subprocess

SANDBOX_IMAGE = 'python:3.12-alpine'

# 後片付けのときに、この教材が起動したコンテナだけを選べるようにする
SANDBOX_LABEL = 'ai-agent-seminar=session09'

docker_cli = shutil.which('docker')
codex_cli = shutil.which('codex')

# host: 追加の依存が無いので常に使えるが、隔離もされない
print('host : 利用できます（隔離はされません）')

# codex: Codex CLI が必要。コンテナ内では Landlock が使えず起動に失敗することもある
if codex_cli is None:
    print('codex: Codex CLI が見つかりません')
else:
    print('codex: 利用できます:', codex_cli)

# docker: CLI とイメージの両方が必要
if docker_cli is None:
    print('docker: Docker CLI が見つかりません')
else:
    image_probe = subprocess.run(
        [docker_cli, 'image', 'inspect', SANDBOX_IMAGE],
        capture_output=True,
        text=True,
    )
    if image_probe.returncode == 0:
        print('docker: 利用できます:', SANDBOX_IMAGE)
    else:
        print(f'docker: イメージがありません。先に実行してください: docker pull {SANDBOX_IMAGE}')

In [ ]:
# @title 3つのExecutionPolicyの定義
from langchain.agents.middleware import (
    CodexSandboxExecutionPolicy,
    DockerExecutionPolicy,
    HostExecutionPolicy,
)

# どの Policy にも共通する上限。実行時間と、モデルへ返す観測の大きさを抑える
COMMON_LIMITS = {
    # コマンド1件あたりの実行時間の上限。超えるとセッションを再起動する
    'command_timeout': 15.0,
    'startup_timeout': 60.0,
    # モデルへ返す観測を切り詰める
    'max_output_lines': 50,
    'max_output_bytes': 4_000,
}

# --- Docker: 専用コンテナへ隔離する -------------------------------------------
docker_policy = DockerExecutionPolicy(
    image=SANDBOX_IMAGE,
    # ネットワークを無効にする（--network none）
    network_enabled=False,
    # ルートファイルシステムを読み取り専用にする（--read-only）
    read_only_rootfs=True,
    # 非 root ユーザー（nobody）で実行する
    user='65534:65534',
    # 資源の上限
    memory_bytes=256 * 1024 * 1024,
    cpus='0.5',
    # Policy が引数を持たない設定は docker run のオプションで補う
    extra_run_args=(
        f'--label={SANDBOX_LABEL}',
        '--tmpfs', '/tmp:rw,noexec,nosuid,nodev,size=64m,mode=1777',
        '--cap-drop=ALL',
        '--security-opt', 'no-new-privileges=true',
        '--pids-limit=64',
    ),
    **COMMON_LIMITS,
)

# --- Codex: Codex CLI のサンドボックスへ委ねる ---------------------------------
codex_policy = CodexSandboxExecutionPolicy(
    # 'auto' は OS を見て Seatbelt（macOS）と Landlock/seccomp（Linux）を選ぶ
    platform='auto',
    # codex の -c オプションへ渡す設定。ここでは外向き通信を止める
    # 新しい権限プロファイル（[permissions]）を使う設定では、こちらは無視される
    config_overrides={'sandbox_workspace_write.network_access': False},
    # Policy 自体は資源制限を持たないため、cgroup などホスト側の仕組みと併用する
    **COMMON_LIMITS,
)

# --- Host: ホストプロセスでそのまま動かす -------------------------------------
host_policy = HostExecutionPolicy(
    # ファイルもネットワークも隔離されない。守れるのは CPU 時間とメモリだけ
    cpu_time_seconds=10,
    memory_bytes=1024 * 1024 * 1024,
    **COMMON_LIMITS,
)

In [ ]:
# @title ShellToolMiddlewareの設定
from dataclasses import dataclass
from typing import Any

from langchain.agents.middleware import ShellToolMiddleware


@dataclass
class ShellSessionSpec:
    """ExecutionPolicy ごとに変わるシェルセッションの設定。

    シェルの実体（bash / sh）と作業ディレクトリの初期化は Policy によって変わる。
    note はサンドボックスの制約をモデルへ伝えるための説明文。
    """

    policy: Any
    shell_command: str
    startup_commands: tuple[str, ...]
    note: str


SHELL_SESSION_SPECS = {
    'docker': ShellSessionSpec(
        policy=docker_policy,
        # alpine イメージに bash は無い。既定の /bin/bash では起動に失敗する
        shell_command='/bin/sh',
        # 書き込める /tmp を作業ディレクトリにする。永続セッションなので cd は引き継がれる
        startup_commands=('cd /tmp',),
        note="""shell Tool は隔離コンテナ内の /bin/sh セッションで、次の制約がある。
- bash は無い。POSIX sh の構文で書く
- 書き込めるのは /tmp だけ。ルートファイルシステムは読み取り専用
- 外部ネットワークへは接続できない""",
    ),
    'codex': ShellSessionSpec(
        policy=codex_policy,
        shell_command='/bin/bash',
        # 作業ディレクトリは一時ディレクトリが割り当てられるので cd は不要
        startup_commands=(),
        note="""shell Tool は Codex CLI のサンドボックス内の /bin/bash セッション。
- ホスト上で動くが、syscall とファイルアクセスが制限されている
- 書き込めるのは作業ディレクトリと /tmp だけ
- 外部ネットワークへは接続できない""",
    ),
    'host': ShellSessionSpec(
        policy=host_policy,
        shell_command='/bin/bash',
        startup_commands=(),
        note="""shell Tool はホスト上の /bin/bash セッションで、隔離されていない。
- 作業ディレクトリは一時ディレクトリ。その外のファイルは変更しない
- 外部ネットワークへ接続できる""",
    ),
}

execution_policy_name = 'docker' #@param ['docker', 'codex', 'host']
spec = SHELL_SESSION_SPECS[execution_policy_name]

# 前提のCLIが無いまま進むと、グラフ実行の途中でセッション起動に失敗する
required_cli = {'docker': docker_cli, 'codex': codex_cli}.get(execution_policy_name)
if execution_policy_name in ('docker', 'codex') and required_cli is None:
    raise RuntimeError(
        f'{execution_policy_name} を使うにはCLIが必要です。'
        '先のセルの確認結果を見て、利用できるものを選んでください。'
    )

shell_middleware = ShellToolMiddleware(
    execution_policy=spec.policy,
    shell_command=spec.shell_command,
    startup_commands=spec.startup_commands,
    # workspace_root を渡さないため、ホストの作業ディレクトリは bind mount されない
    # env を渡さないため、APIキーなどの環境変数もセッションへ渡らない
)

print('ExecutionPolicy:', type(spec.policy).__name__)
print('追加されるTool:', [tool.name for tool in shell_middleware.tools])

In [ ]:
# @title ホスト実行Toolの除外と承認対象の更新
# ホストでそのまま実行される Tool は外す。残すと sandbox を迂回できてしまう
HOST_EXECUTION_TOOLS = {'run_command', 'python_repl'}
SANDBOX_TOOLS = [tool for tool in DEFAULT_TOOLS if tool.name not in HOST_EXECUTION_TOOLS]

# 承認対象に shell を追加する
sandbox_hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        'shell': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'write_file': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'file_delete': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'read_file': False,
    },
    description_prefix='Toolの実行には承認が必要',
)

print('除外したTool:', sorted(HOST_EXECUTION_TOOLS))
print('エージェントへ渡すTool:', [tool.name for tool in SANDBOX_TOOLS] + ['shell'])

In [ ]:
# @title 長期記憶グラフへサンドボックス付きエージェントを組み込む
@dynamic_prompt
def sandbox_system_prompt(request: ModelRequest) -> str:
    memories = request.state.get('memories', [])
    # 選んだ ExecutionPolicy の制約をモデルへ伝える
    prompt = f"""あなたはAIエージェントです。
必要な場合はToolを利用してください。

コマンドやコードの実行には shell Tool だけを使うこと。
出力が改行で終わるコマンドを使うこと。
{spec.note}
"""
    if memories:
        memory_text = '\n'.join(f'- {memory}' for memory in memories)
        prompt += f"""
参考になる長期記憶:
{memory_text}
"""
    return prompt


sandbox_agent = create_agent(
    model=model,
    tools=SANDBOX_TOOLS,
    state_schema=LongTermMemoryState,
    middleware=[sandbox_system_prompt, shell_middleware, sandbox_hitl_middleware],
)

# 長期記憶グラフの構造は前半と同じ。agent ノードだけ差し替える
sandbox_builder = StateGraph(
    LongTermMemoryState,
    context_schema=Context,
)
sandbox_builder.add_node('load_memory', load_memory)
sandbox_builder.add_node('agent', sandbox_agent)
sandbox_builder.add_node('extract_memory', make_extract_memory(model))
sandbox_builder.add_node('validate_memory', validate_memory)
sandbox_builder.add_node('write_memory', write_memory)

sandbox_builder.add_edge(START, 'load_memory')
sandbox_builder.add_edge('load_memory', 'agent')
sandbox_builder.add_edge('agent', 'extract_memory')
sandbox_builder.add_edge('extract_memory', 'validate_memory')
sandbox_builder.add_edge('validate_memory', 'write_memory')
sandbox_builder.add_edge('write_memory', END)

# 長期記憶の Store は前半と共有する
graph_with_sandbox = sandbox_builder.compile(
    store=long_term_store,
    checkpointer=InMemorySaver(),
)

display(
    Image(
        graph_with_sandbox.get_graph(xray=2).draw_mermaid_png(curve_style=CurveStyle.NATURAL)
    )
)

In [ ]:
# @title shellの実行を承認する
sandbox_config = {
    'configurable': {'thread_id': f'sandbox-agent-{uuid4()}'}
}

pending_shell = graph_with_sandbox.invoke(
    {
        'messages': [
            HumanMessage(
                content='1から100までの素数の個数を、shell Toolで計算してください。'
            )
        ]
    },
    config=sandbox_config,
    context=hitl_context,
)

review = pending_shell['__interrupt__'][0].value
print(json.dumps(review, ensure_ascii=False, indent=2))

In [ ]:
# @title 承認してサンドボックス内で実行する
def approve_all(state, config, *, max_rounds=5):
    """承認を求められる限り approve で再開する。

    ReActループはツールを何度も呼ぶため、1回の依頼で承認要求が複数回発生しうる。
    実行が終わると `__interrupt__` が無くなる。
    """
    for _ in range(max_rounds):
        if '__interrupt__' not in state:
            return state
        request = state['__interrupt__'][0].value['action_requests'][0]
        print('承認したコマンド:', request['args'].get('command'))
        state = graph_with_sandbox.invoke(
            Command(resume={'decisions': [{'type': 'approve'}]}),
            config=config,
            context=hitl_context,
        )
    raise RuntimeError(f'承認が{max_rounds}回を超えました')


approved_shell = approve_all(pending_shell, sandbox_config)

# ツールの観測（サンドボックスの出力）と最終回答を確認する
for message in approved_shell['messages'][-2:]:
    message.pretty_print()

In [ ]:
# @title コマンドを修正して実行する
# レビュー担当者が実行内容を確定させる。承認した文字列がそのままシェルへ渡る。
# 同じコマンドでも、選んだ ExecutionPolicy によって結果が変わる
SANDBOX_PROBE_COMMAND = (
    'id; '
    'echo sandboxed > /tmp/result.txt && echo "tmp: $(cat /tmp/result.txt)"; '
    "echo blocked > /blocked.txt 2>/dev/null "
    "&& echo 'rootfs: 書き込めた' || echo 'rootfs: 書き込み不可'; "
    'wget -q -T 3 -O- https://example.com > /dev/null 2>&1 '
    "&& echo 'network: 到達' || echo 'network: 到達不可'"
)

edit_config = {
    'configurable': {'thread_id': f'sandbox-agent-{uuid4()}'}
}

pending_edit = graph_with_sandbox.invoke(
    {
        'messages': [
            HumanMessage(
                content=(
                    'shell Toolで、このサンドボックスの実行ユーザーと'
                    'アクセスできる範囲を確認してください。'
                )
            )
        ]
    },
    config=edit_config,
    context=hitl_context,
)

print('モデルの提案:', pending_edit['__interrupt__'][0].value['action_requests'][0]['args'])

edited_shell = graph_with_sandbox.invoke(
    Command(
        resume={
            'decisions': [{
                'type': 'edit',
                'edited_action': {
                    'name': 'shell',
                    'args': {'command': SANDBOX_PROBE_COMMAND},
                },
            }],
        }
    ),
    config=edit_config,
    context=hitl_context,
)

# 修正後もモデルが追加のコマンドを求めることがあるため、残りは承認で進める
edited_shell = approve_all(edited_shell, edit_config)

for message in edited_shell['messages'][-2:]:
    message.pretty_print()

In [ ]:
# @title サンドボックスコンテナの後片付け
# DockerExecutionPolicy を使ったときだけ意味がある。
# 中断したまま再開しなかった実行のコンテナは、参照が切れた時点で片付けられる
import gc

gc.collect()

if docker_cli is None:
    print('Docker CLI が無いため、後片付けするコンテナはありません')
else:
    leftover = subprocess.run(
        [docker_cli, 'ps', '--quiet', '--filter', f'label={SANDBOX_LABEL}'],
        capture_output=True,
        text=True,
    ).stdout.split()

    if leftover:
        subprocess.run(
            [docker_cli, 'rm', '--force', *leftover],
            capture_output=True,
            text=True,
        )

    print('強制削除したサンドボックスコンテナ:', len(leftover))

### 実装の要点

- **多層防御が1つのグラフに収まる**: `HumanInTheLoopMiddleware` が「実行してよいか」を人間へ問い、ExecutionPolicy が「実行中に何ができるか」を制限する。`edit` の例で確認したように、人間が承認・修正した内容でも、`docker` ではルートファイルシステムへの書き込みと外部通信がコンテナ側で失敗する
- **迂回路を残さない**: `run_command` や `python_repl` のようにホストで動く Tool を残すと、モデルは制限の緩い経路を選べてしまう。コマンド実行の入口を `shell` だけに絞る
- **設定を固定する**: エージェントが渡せるのはコマンド文字列だけ。イメージ、マウント、ネットワーク、権限、資源は ExecutionPolicy 側で決める
- **隔離の強さだけを差し替える**: Tool の名前と引数は Policy によらず同じなので、`create_agent()` へ渡す `ShellToolMiddleware` を作り直すだけで実行環境を切り替えられる。エージェントの実装やプロンプトの構造は変えなくてよい
- **ホストへフォールバックしない**: Docker CLI が無ければセッションの起動を失敗させ、同じコマンドをホストで実行しない
- **環境を伝える**: サンドボックスの制約（`bash` が無い、書けるのは `/tmp` だけ、ネットワーク不可）をシステムプロンプトへ書くと、失敗するコマンドの生成と再試行が減る
- **秘密情報を渡さない**: `ShellToolMiddleware` に `env` を渡さなければ、コンテナへ API キーなどの環境変数は渡らない

### 注意点

- **永続セッションの副作用**: セッションは実行をまたいで状態を保つため、あるコマンドが作った `/tmp` のファイルや `cd` の結果が後続のコマンドへ影響する。`restart` でセッションを作り直せる
- **出力の末尾に改行が必要**: 完了検知に行単位のマーカーを使うため、`printf %s ...` のように改行で終わらない出力はマーカーと同じ行に混ざり、`command_timeout` まで待ってタイムアウトする。上限は短く設定する
- **中断中のコンテナ**: `interrupt()` で停止した実行のセッションはチェックポイントに保存されないため、再開時には新しいコンテナが起動する。古いコンテナは参照が切れた時点で片付けられるので、ノートブックを繰り返し実行する場合は後片付けのセルを使う

本番用途へ発展させる場合は、イメージをタグではなく digest で固定し、脆弱性スキャン、同時実行数とディスク容量の制限、監査ログ、成果物の検疫、外向き通信のプロキシ制御を追加する。Docker デーモンを操作できる外側のランナー自体は強い権限を持つため、信頼できるサービスとしてエージェントから分離する。